# 🍳 Notebook 4 — Combined Filters: The Chef's Recipe

> **The vibe:** Temperature = how hot the oven is. Top-k = which ingredients you're allowed to use. Top-p = how much of each ingredient based on taste.  
> **Key word:** ORDER — add ingredients in the wrong order and the dish is ruined.  
> **Time:** ~12 minutes

---

## 🎯 The Pipeline (Order is Law)

```
Raw logits
    ↓  ① Temperature  →  reshape the whole distribution
    ↓  ② Top-k        →  hard cutoff by rank
    ↓  ③ Top-p        →  adaptive cut by probability
    ↓  Sample!
```

**Changing the order changes the output.** This is not negotiable.


In [ ]:
# ── Setup ────────────────────────────────────────────────────────────
import numpy as np, matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
plt.rcParams.update({'figure.facecolor':'#0d1117','axes.facecolor':'#161b22',
    'axes.edgecolor':'#30363d','text.color':'#e6edf3','axes.labelcolor':'#8b949e',
    'xtick.color':'#8b949e','ytick.color':'#8b949e','grid.color':'#21262d',
    'axes.grid':True,'grid.linewidth':0.5,'font.family':'monospace'})

TOKENS = ['approve','reject','review','escalate','delay','audit','optimize','notify','assign','close']
LOGITS = np.array([2.2,1.8,1.4,0.9,0.2,0.1,-0.3,-0.6,-0.8,-1.0])

def softmax(x): e=np.exp(x-x.max()); return e/e.sum()
def apply_temp(logits, T): return softmax(logits / T)
def apply_top_k(probs, k):
    if k <= 0 or k >= len(probs): return probs.copy()
    f = np.zeros_like(probs); top = np.argsort(probs)[-k:]; f[top] = probs[top]
    return f / f.sum()
def apply_top_p(probs, p):
    si = np.argsort(probs)[::-1]; cs = np.cumsum(probs[si])
    cut = np.searchsorted(cs, p) + 1; ni = si[:cut]
    f = np.zeros_like(probs); f[ni] = probs[ni]; return f / f.sum(), cut

def full_pipeline(logits, T=1.0, k=0, p=1.0):
    step1 = apply_temp(logits, T)
    step2 = apply_top_k(step1, k) if k > 0 else step1.copy()
    step3, nucleus_size = apply_top_p(step2, p)
    return step1, step2, step3, nucleus_size

print("✅ Setup done! Four preset 'recipes' ready:")
recipes = [
    ('🎯 Precise',   0.2,  40, 0.9,  'Code, facts'),
    ('⚖️  Balanced', 0.7,  50, 0.9,  'General chat'),
    ('🎨 Creative',  1.0, 100, 0.95, 'Storytelling'),
    ('🚀 Explorer',  1.2,   0, 1.0,  'Brainstorming'),
]
for name, T, k, p, use in recipes:
    _, _, final, ns = full_pipeline(LOGITS, T, k, p)
    ent = -np.sum(final * np.log(final + 1e-12))
    print(f"  {name:15s} T={T}, k={k:3d}, p={p} → entropy={ent:.2f}, nucleus={ns}")


## 🎬 Graph 1 — Watch the Pipeline Transform a Distribution

This shows the **same distribution** flowing through all 4 stages.

Watch how each stage progressively narrows the candidates.  
The "eligible" count at the top of each panel is your key number.


In [ ]:
# ── Graph 1: The Pipeline Step by Step ──────────────────────────────
T_demo, K_demo, P_demo = 0.7, 5, 0.9
s1, s2, s3, ns = full_pipeline(LOGITS, T_demo, K_demo, P_demo)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle(f'🔬 The Pipeline in Action: T={T_demo}, k={K_demo}, p={P_demo}',
             fontsize=14, fontweight='bold', color='#ffd93d', y=1.02)

stages = [
    ('① Raw Logits',     LOGITS, '#8b949e', '10 raw scores
(not probabilities yet)'),
    (f'② After T={T_demo}', s1,   '#ff9f43', f'10 tokens
(all still eligible)'),
    (f'③ After k={K_demo}',  s2,   '#ffd93d', f'{K_demo} tokens
(hard cutoff by rank)'),
    (f'④ After p={P_demo}',  s3,   '#a8e6cf', f'{ns} tokens
(adaptive nucleus)'),
]
for ax, (title, data, col, subtitle) in zip(axes, stages):
    ax.bar(range(10), data, color=col, alpha=0.85, width=0.7)
    eligible = sum(d > 0 for d in data) if max(data) < 5 else 10
    ax.set_xticks(range(10)); ax.set_xticklabels(TOKENS, rotation=45, ha='right', fontsize=7)
    ax.set_title(f'{title}
{subtitle}', color=col, fontweight='bold', fontsize=10)
    ax.set_ylabel('Value (logit or probability)')
    for i, val in enumerate(data):
        if val > max(data) * 0.1:
            ax.text(i, val, f'{val:.2f}', ha='center', va='bottom', fontsize=6, color='white')

# Arrow between panels
for i in range(3):
    fig.text(0.21 + i*0.2, 0.5, '→', fontsize=30, color='#ffd93d',
             ha='center', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('graph1_combined_pipeline.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f"\n💡 Started with 10 candidates, ended with {ns}.")
print(f"💡 Top-k removed {10 - K_demo}, top-p removed {K_demo - ns} more.")
print(f"💡 Final nucleus: {[TOKENS[i] for i in np.argsort(s3)[::-1][:ns]]}")


## 🎬 Graph 2 — The 4 Recipes Side by Side

Compare all 4 presets at once. Notice:
- Entropy increases from Precise → Explorer  
- Nucleus size increases with creativity  
- The *shape* of each distribution reflects the intent


In [ ]:
# ── Graph 2: Four Recipes ────────────────────────────────────────────
recipes_data = [
    ('🎯 Precise',  0.2,  40, 0.9,  '#ff6b9d'),
    ('⚖️ Balanced', 0.7,  50, 0.9,  '#ffd93d'),
    ('🎨 Creative', 1.0, 100, 0.95, '#a8e6cf'),
    ('🚀 Explorer', 1.2,   0, 1.0,  '#74b9ff'),
]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('🍳  The Four Recipes: Same Ingredients, Very Different Dishes',
             fontsize=14, fontweight='bold', color='#ffd93d', y=1.02)

for ax, (name, T, k, p, col) in zip(axes, recipes_data):
    _, _, final, ns = full_pipeline(LOGITS, T, k, p)
    ent = -np.sum(final * np.log(final + 1e-12))
    ax.bar(range(10), final, color=col, alpha=0.85, width=0.7)
    ax.set_xticks(range(10)); ax.set_xticklabels(TOKENS, rotation=45, ha='right', fontsize=7)
    ax.set_ylim(0, 1.05)
    ax.set_title(f'{name}
T={T}, k={k}, p={p}
H={ent:.2f}, nucleus={ns}',
                 color=col, fontweight='bold', fontsize=9)
    ax.set_ylabel('Probability' if ax==axes[0] else '')
    # Show top 3 tokens
    top3 = np.argsort(final)[::-1][:3]
    for rank, idx in enumerate(top3):
        if final[idx] > 0.01:
            ax.text(idx, final[idx]+0.01, f'#{rank+1}', ha='center', va='bottom',
                   color='white', fontsize=7, fontweight='bold')

plt.tight_layout()
plt.savefig('graph2_combined_recipes.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("\n📊 Entropy progression:")
for name, T, k, p, _ in recipes_data:
    _, _, final, ns = full_pipeline(LOGITS, T, k, p)
    ent = -np.sum(final * np.log(final + 1e-12))
    bar = '█' * int(ent * 5)
    print(f"  {name:15s} {bar:20s} {ent:.2f} bits (nucleus: {ns})")


## 🎬 Graph 3 — The Interaction Heatmap: Temperature × Top-k

This heatmap shows entropy across a grid of T and k values simultaneously.

**Reading the heatmap:**
- **Bright (yellow)** = high entropy = diverse outputs  
- **Dark (purple)** = low entropy = deterministic outputs  
- Moving RIGHT (higher T): entropy increases  
- Moving UP (higher k): entropy increases  
- **The bottom-left corner is always the darkest** = greedy decoding zone


In [ ]:
# ── Graph 3: Interaction Heatmap ─────────────────────────────────────
T_vals  = np.linspace(0.1, 2.0, 25)
k_vals  = [1, 2, 3, 5, 7, 10]
k_labels = [f'k={k}' for k in k_vals]

entropy_grid = np.zeros((len(k_vals), len(T_vals)))
for i, k in enumerate(k_vals):
    for j, T in enumerate(T_vals):
        pr = apply_temp(LOGITS, T)
        pr = apply_top_k(pr, k)
        entropy_grid[i, j] = -np.sum(pr * np.log(pr + 1e-12))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('🗺️  Entropy Map: Temperature × Top-k Interaction',
             fontsize=14, fontweight='bold', color='#ffd93d')

im = ax1.imshow(entropy_grid, aspect='auto', cmap='plasma',
                extent=[T_vals[0], T_vals[-1], -0.5, len(k_vals)-0.5], origin='lower')
plt.colorbar(im, ax=ax1, label='Entropy (bits)', shrink=0.9)
ax1.set_yticks(range(len(k_vals))); ax1.set_yticklabels(k_labels)
ax1.set_xlabel('Temperature →'); ax1.set_title('Entropy Landscape
(bright = diverse, dark = precise)',
                                                color='#ffd93d', fontweight='bold')
# Mark the 4 presets
preset_positions = [(0.2, 0), (0.7, 4), (1.0, 5), (1.2, 5)]  # (T, k_idx)
preset_names2 = ['Precise','Balanced','Creative','Explorer']
preset_cols2  = ['#ff6b9d','#ffd93d','#a8e6cf','#74b9ff']
for (T_pos, k_idx), pname, pcol in zip(preset_positions, preset_names2, preset_cols2):
    ax1.scatter(T_pos, k_idx, s=200, color=pcol, zorder=5, edgecolors='white', lw=2)
    ax1.text(T_pos+0.05, k_idx+0.15, pname, color=pcol, fontsize=9, fontweight='bold')

# Panel 2: Entropy vs T for each k
for i, (k, col2) in enumerate(zip(k_vals, ['#ff6b9d','#ff9f43','#ffd93d','#a8e6cf','#74b9ff','#a29bfe'])):
    ax2.plot(T_vals, entropy_grid[i], color=col2, lw=2, label=f'k={k}')
ax2.set_xlabel('Temperature'); ax2.set_ylabel('Entropy (bits)')
ax2.set_title('Entropy vs T for each k value
(lines converge at high T = k stops mattering)',
              color='#ffd93d', fontweight='bold')
ax2.legend(fontsize=9, ncol=2)
ax2.axvline(0.7, color='white', lw=1, ls=':', alpha=0.5)

plt.tight_layout()
plt.savefig('graph3_combined_heatmap.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("\n💡 Notice how the lines converge at high T:")
print("💡 When T is large enough, k stops mattering — the distribution is flat either way.")


## 🎮 Graph 4 — Build Your Own Recipe

Set your own T, k, p values and see exactly what you get.

**Experiment ideas:**
- Try T=0.1, k=1, p=1.0 → maximum precision, greedy decoding
- Try T=0.7, k=3, p=0.5 → balanced but tight
- Try T=1.5, k=0, p=0.9 → creative with nucleus safety


In [ ]:
# ── Graph 4: Build Your Own Recipe ──────────────────────────────────
# 🍳 YOUR RECIPE — CHANGE THESE!
MY_T = 0.7   # ← Temperature (0.1 to 2.0)
MY_K = 5     # ← Top-k (0=disabled, 1=greedy, 50=broad)
MY_P = 0.9   # ← Top-p (0.5=tight, 0.9=standard, 1.0=off)

# ─────────────────────────────────────────────────────────────────────
s1, s2, s3, ns = full_pipeline(LOGITS, MY_T, MY_K, MY_P)
ent = -np.sum(s3 * np.log(s3 + 1e-12))

fig = plt.figure(figsize=(16, 6))
gs  = gridspec.GridSpec(1, 4, figure=fig, wspace=0.4)
fig.patch.set_facecolor('#0d1117')
fig.suptitle(f'🍳 YOUR Recipe: T={MY_T}, k={MY_K}, p={MY_P}  →  entropy={ent:.2f}, nucleus={ns}',
             fontsize=13, fontweight='bold', color='#ffd93d')

stages2 = [(LOGITS,'Raw','#8b949e'),(s1,f'After T={MY_T}','#ff9f43'),
           (s2,f'After k={MY_K}','#ffd93d'),(s3,f'After p={MY_P}','#a8e6cf')]
for i, (data, label, col) in enumerate(stages2):
    ax = fig.add_subplot(gs[i])
    ax.bar(range(10), data, color=col, alpha=0.85, width=0.7)
    ax.set_xticks(range(10)); ax.set_xticklabels(TOKENS, rotation=45, ha='right', fontsize=7)
    n_elig = sum(d > 0 for d in data) if i > 0 else 10
    ax.set_title(f'{label}
{"(logit values)" if i==0 else f"{n_elig} eligible"}',
                color=col, fontweight='bold', fontsize=10)

zone = '🎯 Precise' if ent < 1.0 else '⚖️ Balanced' if ent < 1.8 else '🎨 Creative' if ent < 2.5 else '🚀 Explorer'
top3_final = [TOKENS[i] for i in np.argsort(s3)[::-1][:3]]
print(f"\n🍳 Your Recipe Results:")
print(f"   Entropy:  {ent:.3f} bits  {zone}")
print(f"   Nucleus:  {ns} tokens")
print(f"   Top 3:    {top3_final}")
print(f"   P(#1):    {s3.max():.1%}")

plt.savefig('graph4_combined_playground.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()


## ✅ Notebook 4 Complete

| Concept | In One Line |
|---|---|
| Pipeline | Temperature → Top-k → Top-p; order is fixed and matters |
| Interaction | High T + low p can cancel out; conservative settings compound |
| Heatmap | Entropy map shows where each parameter dominates |
| Recipes | Precise (T=0.2), Balanced (T=0.7), Creative (T=1.0), Explorer (T=1.2) |

> **🍳 Chef metaphor holds:** Same ingredients, different proportions and order = completely different dishes.

**Next → `05_repetition.ipynb`: The Anti-Echo Chamber 🔁**
